Introdução do projeto para dar nome aos bois

In [1]:
import os
import torch

PASTA_PROJETO = os.getenv("PASTA_PROJETO", "/content/drive/MyDrive/projeto_dionisio")
os.makedirs(PASTA_PROJETO, exist_ok=True)

CAMINHO_CHROMA = os.path.join(PASTA_PROJETO, "chroma_db")
CAMINHO_QA_BASE = os.path.join(PASTA_PROJETO, "catalogo_qa_base.json")
CAMINHO_FAQ_CLIENTES = os.path.join(PASTA_PROJETO, "faq_customizado_por_cliente.json")
CAMINHO_LOG_REPROVADAS = os.path.join(PASTA_PROJETO, "log_respostas_reprovadas.json")

MODELO_EMBEDDING = "BAAI/bge-m3"
URL_ALVO_PADRAO = "https://pt.wikipedia.org/wiki/Festa"

# Dados reais extraídos do benchmark de GPUs do Google Colab (Chris McCormick)
PERFIS_GPU_COLAB = {
    "T4":   {"unidades_hora": 1.84, "vram_util_gb": 15.0, "arquitetura": "Turing",       "suporta_bf16": False, "num_ctx_seguro": 1024},
    "V100": {"unidades_hora": 4.91, "vram_util_gb": 16.0, "arquitetura": "Volta",        "suporta_bf16": False, "num_ctx_seguro": 2048},
    "L4":   {"unidades_hora": 4.82, "vram_util_gb": 22.5, "arquitetura": "Ada Lovelace", "suporta_bf16": True,  "num_ctx_seguro": 4096},
    "A100": {"unidades_hora": 11.77,"vram_util_gb": 40.0, "arquitetura": "Ampere",       "suporta_bf16": True,  "num_ctx_seguro": 4096},
}

def detectar_configuracao_hardware(creditos_disponiveis: float = 200.0) -> dict:
    """
    Detecta a GPU ativa no Colab e ajusta automaticamente:
    - Tipo de ponto flutuante: bfloat16 (L4/A100) vs float16 (T4/V100).
    - Implementação de atenção: 'sdpa' (compatível com T4 e tão rápido quanto FlashAttention v2).
    - Janela de contexto (num_ctx) e modelo para respeitar os 15 GB úteis da T4 (descontado 1 GB de ECC).
    """
    if not torch.cuda.is_available():
        return {
            "dispositivo": "cpu",
            "nome_gpu": "CPU",
            "dtype": torch.float32,
            "attn_implementation": "sdpa",
            "modelo_llm": "qwen3:8b",
            "num_ctx": 1024,
            "unidades_hora": 0.10,
            "horas_estimadas_200_creditos": round(creditos_disponiveis / 0.10, 1)
        }

    nome_placa = torch.cuda.get_device_name(0)
    perfil = PERFIS_GPU_COLAB["T4"] # Padrão conservador (T4)
    chave_detectada = "T4"

    for chave, dados in PERFIS_GPU_COLAB.items():
        if chave in nome_placa.upper():
            perfil = dados
            chave_detectada = chave
            break

    # Seleciona bfloat16 apenas em GPUs mais novas (L4 e A100); mantém float16 na T4 e V100
    dtype_ideal = torch.bfloat16 if perfil["suporta_bf16"] else torch.float16

    # Na T4 (apenas 15 GB úteis devido a 1 GB do ECC), podemos usar qwen3:8b no desenvolvimento
    # ou qwen3:14b com sequência controlada (1024) para não estourar a VRAM
    modelo_padrao = os.getenv("MODELO_LLM", "qwen3:14b")
    horas_restantes = round(creditos_disponiveis / perfil["unidades_hora"], 1)

    info = {
        "dispositivo": "cuda",
        "nome_gpu": f"{nome_placa} ({chave_detectada})",
        "vram_util_gb": perfil["vram_util_gb"],
        "dtype": dtype_ideal,
        "attn_implementation": "sdpa", # Evita erro do flash_attention_2 na T4
        "modelo_llm": modelo_padrao,
        "num_ctx": perfil["num_ctx_seguro"],
        "unidades_hora": perfil["unidades_hora"],
        "horas_estimadas_200_creditos": horas_restantes
    }

    print(f"=== DIAGNÓSTICO DE GPU DO COLAB ===")
    print(f"GPU Detectada: {info['nome_gpu']} | VRAM Útil Real: {info['vram_util_gb']} GB")
    print(f"Precisão Selecionada: {info['dtype']} | Atenção: {info['attn_implementation']} (PyTorch 2.2+)")
    print(f"Custo Estimado: {info['unidades_hora']} unidades/h -> Seus {creditos_disponiveis} créditos rendem ~{horas_restantes} horas.")
    print(f"Janela de Contexto Segura (num_ctx): {info['num_ctx']} tokens")
    print(f"===================================")
    return info

HARDWARE_ATUAL = detectar_configuracao_hardware()
MODELO_LLM = HARDWARE_ATUAL["modelo_llm"]
NUM_CTX_SEGURO = HARDWARE_ATUAL["num_ctx"]


=== DIAGNÓSTICO DE GPU DO COLAB ===
GPU Detectada: NVIDIA L4 (L4) | VRAM Útil Real: 22.5 GB
Precisão Selecionada: torch.bfloat16 | Atenção: sdpa (PyTorch 2.2+)
Custo Estimado: 4.82 unidades/h -> Seus 200.0 créditos rendem ~41.5 horas.
Janela de Contexto Segura (num_ctx): 4096 tokens


Montar o Google Drive e Instalar Dependências
Rode esta célula primeiro para configurar o ambiente, instalar o navegador invisível do Crawl4AI e o Ollama.

In [2]:
!pip install crawl4ai chromadb sentence-transformers fastapi uvicorn ollama

In [3]:
import importlib

dependencias = [
    "torch",
    "crawl4ai",
    "chromadb",
    "sentence_transformers",
    "fastapi",
    "uvicorn",
    "ollama",
    "pydantic"
]

print("=== STATUS DAS DEPENDÊNCIAS ===")
for dep in dependencias:
    try:
        mod = importlib.import_module(dep)
        version = getattr(mod, "__version__", "Desconhecida")
        print(f"✅ {dep.ljust(22)} -> OK (versão: {version})")
    except ImportError:
        print(f"❌ {dep.ljust(22)} -> NÃO ENCONTRADO!")
print("=================================")

=== STATUS DAS DEPENDÊNCIAS ===
✅ torch                  -> OK (versão: 2.11.0+cu130)
✅ crawl4ai               -> OK (versão: <module 'crawl4ai.__version__' from '/usr/local/lib/python3.13/dist-packages/crawl4ai/__version__.py'>)
✅ chromadb               -> OK (versão: 1.5.9)
✅ sentence_transformers  -> OK (versão: 5.7.0)
✅ fastapi                -> OK (versão: 0.141.1)
✅ uvicorn                -> OK (versão: 0.54.0)
✅ ollama                 -> OK (versão: Desconhecida)
✅ pydantic               -> OK (versão: 2.13.5)


In [4]:
import asyncio
import re
import nest_asyncio
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode

nest_asyncio.apply()

async def extrair_markdown_do_site(url: str) -> str:
    """Acessa o HTML do site da plataforma de festas, limpa ruídos e converte em Markdown."""
    browser_cfg = BrowserConfig(headless=True)
    run_cfg = CrawlerRunConfig(
        cache_mode=CacheMode.BYPASS,
        excluded_tags=["nav", "footer", "header", "script", "style", "aside", "noscript"],
        remove_overlay_elements=True,
    )
    async with AsyncWebCrawler(config=browser_cfg) as crawler:
        resultado = await crawler.arun(url=url, config=run_cfg)
        if not resultado.success:
            raise RuntimeError(f"Erro ao acessar {url}: {resultado.error_message}")
        return resultado.markdown

def dividir_markdown_em_chunks(markdown_texto: str, tamanho_max: int = 1200) -> list[dict]:
    """
    Divide o Markdown em blocos de até 1200 caracteres (~300-400 tokens),
    garantindo que fiquem dentro do limite ótimo de 1024 tokens de sequência na GPU T4.
    """
    secoes = re.split(r'(?m)^(?=#{1,3}\s)', markdown_texto)
    chunks = []

    for idx, secao in enumerate(secoes):
        secao = secao.strip()
        if len(secao) < 60:
            continue

        linhas = secao.split('\n')
        titulo = linhas[0].lstrip('#').strip() if linhas[0].startswith('#') else f"Seção {idx+1}"

        if len(secao) <= tamanho_max:
            chunks.append({"id": f"chunk_{len(chunks)}", "titulo": titulo, "conteudo": secao})
        else:
            paragrafos = secao.split('\n\n')
            bloco_atual = f"# {titulo}\n\n"
            for p in paragrafos:
                if len(bloco_atual) + len(p) <= tamanho_max:
                    bloco_atual += p + "\n\n"
                else:
                    chunks.append({"id": f"chunk_{len(chunks)}", "titulo": titulo, "conteudo": bloco_atual.strip()})
                    bloco_atual = f"# {titulo} (continuação)\n\n{p}\n\n"
            if len(bloco_atual.strip()) > 60:
                chunks.append({"id": f"chunk_{len(chunks)}", "titulo": titulo, "conteudo": bloco_atual.strip()})

    return chunks


Carrega o BAAI/bge-m3 usando

In [5]:
import chromadb
from sentence_transformers import SentenceTransformer

def carregar_modelo_embedding():
    """
    Carrega o BAAI/bge-m3 usando:
    - 'sdpa' (Squared Dot Product Attention do PyTorch 2.2+), que funciona na T4 sem dar o erro do flash_attention_2.
    - bfloat16 automático na L4/A100 ou float16 na T4/V100.
    """
    dispositivo = HARDWARE_ATUAL["dispositivo"]
    kwargs_modelo = {}
    if dispositivo == "cuda":
        kwargs_modelo = {
            "torch_dtype": HARDWARE_ATUAL["dtype"],
            "attn_implementation": HARDWARE_ATUAL["attn_implementation"]
        }

    return SentenceTransformer(
        MODELO_EMBEDDING,
        device=dispositivo,
        model_kwargs=kwargs_modelo
    )

def inicializar_chromadb():
    """Conecta ao ChromaDB persistente no Google Drive."""
    cliente = chromadb.PersistentClient(path=CAMINHO_CHROMA)
    colecao_site = cliente.get_or_create_collection(
        name="conteudo_site_rag",
        metadata={"hnsw:space": "cosine"}
    )
    colecao_cache = cliente.get_or_create_collection(
        name="cache_respostas_aprovadas",
        metadata={"hnsw:space": "cosine"}
    )
    return cliente, colecao_site, colecao_cache

def indexar_chunks_no_chroma(chunks_site: list[dict], url_origem: str, modelo_emb, colecao_site):
    """Vetoriza e salva todos os blocos do site ajustando o batch_size conforme a VRAM."""
    textos = [c["conteudo"] for c in chunks_site]
    ids = [c["id"] for c in chunks_site]
    metadados = [{"titulo": c["titulo"], "url": url_origem} for c in chunks_site]

    # Usa batch menor na T4 (15 GB) e maior na L4 (22.5 GB) / A100 (40 GB)
    lote = 4 if HARDWARE_ATUAL.get("vram_util_gb", 15) <= 15.0 else 16

    vetores = modelo_emb.encode(
        textos,
        batch_size=lote,
        normalize_embeddings=True,
        show_progress_bar=True
    ).tolist()

    colecao_site.upsert(
        ids=ids,
        documents=textos,
        embeddings=vetores,
        metadatas=metadados
    )
    print(f"Sucesso! {colecao_site.count()} blocos indexados com batch_size={lote}.")


Extrai todas as perguntas e respostas possíveis ajustando num_ctx para a VRAM real da GPU.

In [6]:
import json
import ollama
from pydantic import BaseModel, Field

class ParQA(BaseModel):
    pergunta: str = Field(description="Pergunta clara que um cliente faria sobre a plataforma ou organização da festa")
    resposta_base: str = Field(description="Resposta fiel baseada exclusivamente no conteúdo do site")
    categoria: str = Field(description="Categoria (ex: Orçamento, Fornecedores, Convidados, Funcionamento da Plataforma)")

class ListaQA(BaseModel):
    itens: list[ParQA]

class RespostaCustomizada(BaseModel):
    pergunta_adaptada: str = Field(description="A pergunta reescrita para o tipo de festa/evento do cliente")
    resposta_personalizada: str = Field(description="Resposta adaptada ao perfil da festa do cliente mantendo fidelidade às regras do site")
    dicas_praticas_evento: list[str] = Field(description="2 a 3 passos práticos para ajudar o cliente a organizar sua festa na plataforma")

def extrair_qa_do_bloco(bloco_texto: str) -> list[dict]:
    """Extrai todas as perguntas e respostas possíveis ajustando num_ctx para a VRAM real da GPU."""
    prompt_sistema = (
        "Você é um especialista na plataforma de festas Dionísio. "
        "Leia o trecho do site e extraia TODAS as perguntas e respostas relevantes que um usuário "
        "teria sobre o funcionamento da plataforma e sobre como organizar sua festa.\n"
        "Regras:\n"
        "1. Use APENAS informações presentes no trecho.\n"
        "2. Escreva em português claro.\n"
        "3. Retorne estritamente em JSON. /no_think"
    )

    resposta = ollama.chat(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": f"Extraia todas as perguntas e respostas deste trecho:\n\n{bloco_texto} /no_think"}
        ],
        format=ListaQA.model_json_schema(),
        options={"temperature": 0.2, "num_ctx": NUM_CTX_SEGURO}
    )
    dados = json.loads(resposta["message"]["content"])
    return dados.get("itens", [])

def buscar_contexto_rag(consulta: str, modelo_emb, colecao_site, top_k: int = 2) -> str:
    """Recupera os trechos mais relevantes do site no ChromaDB."""
    vetor_busca = modelo_emb.encode([consulta], normalize_embeddings=True).tolist()
    resultados = colecao_site.query(query_embeddings=vetor_busca, n_results=top_k)
    documentos = resultados["documents"][0] if resultados["documents"] else []
    return "\n\n---\n\n".join(documentos)

def customizar_qa_para_cliente(pergunta_base: str, resposta_base: str, perfil_cliente: dict, modelo_emb, colecao_site) -> dict:
    """Adapta a resposta via RAG para o perfil específico da festa/cliente."""
    contexto_site = buscar_contexto_rag(pergunta_base, modelo_emb, colecao_site, top_k=2)

    prompt_sistema = (
        "Você é o assistente inteligente da plataforma de festas Dionísio. "
        "Sua tarefa é responder à dúvida do cliente combinando a Resposta Base, o Contexto do Site (RAG) "
        "e o Perfil do Cliente/Evento.\n"
        "Regras:\n"
        "1. Adapte exemplos e linguagem para o tipo de festa, estilo e objetivo do cliente.\n"
        "2. NUNCA invente funcionalidades ou preços que não estejam no Contexto do site.\n"
        "3. Responda estritamente em JSON. /no_think"
    )

    prompt_usuario = f"""
PERFIL DO CLIENTE / FESTA:
- Nome: {perfil_cliente['nome']}
- Tipo de Evento/Segmento: {perfil_cliente['segmento']}
- Experiência com Eventos: {perfil_cliente['nivel_tecnico']}
- Tom de Voz Desejado: {perfil_cliente['tom_de_voz']}
- Objetivo Principal: {perfil_cliente['objetivo']}

PERGUNTA DO CLIENTE: {pergunta_base}
RESPOSTA OFICIAL BASE: {resposta_base}

CONTEXTO DO SITE (RAG):
{contexto_site}
/no_think
"""
    resposta = ollama.chat(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": prompt_usuario}
        ],
        format=RespostaCustomizada.model_json_schema(),
        options={"temperature": 0.3, "num_ctx": NUM_CTX_SEGURO}
    )
    return json.loads(resposta["message"]["content"])


API Projeto Dionísio - RAG & Feedback Loop

In [7]:
import os
import uuid
import json
from datetime import datetime
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# As funções e variáveis já estão na memória do Colab pelas células anteriores.

modelo_embedding = carregar_modelo_embedding()
_, colecao_site, colecao_cache = inicializar_chromadb()
pendentes_feedback = {}

app = FastAPI(title="API Projeto Dionísio - RAG & Feedback Loop")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class PerfilClienteInput(BaseModel):
    id: str
    nome: str
    segmento: str
    nivel_tecnico: str
    tom_de_voz: str
    objetivo: str

class RequisicaoPergunta(BaseModel):
    pergunta: str
    resposta_base: Optional[str] = ""
    perfil_cliente: PerfilClienteInput

class RequisicaoFeedback(BaseModel):
    interacao_id: str
    aprovado: bool
    comentario_cliente: Optional[str] = ""

@app.get("/status_gpu")
def endpoint_status_gpu():
    """Retorna informações da GPU ativa e consumo de unidades de computação por hora."""
    return {
        "gpu": HARDWARE_ATUAL["nome_gpu"],
        "unidades_por_hora": HARDWARE_ATUAL["unidades_hora"],
        "horas_estimadas_200_creditos": HARDWARE_ATUAL["horas_estimadas_200_creditos"],
        "respostas_aprovadas_no_cache": colecao_cache.count()
    }

@app.post("/perguntar")
def endpoint_perguntar(req: RequisicaoPergunta):
    cliente_id = req.perfil_cliente.id
    pergunta = req.pergunta.strip()

    # 1. Consulta o Cache Semântico de respostas já aprovadas com 👍 (>= 90% de similaridade)
    if colecao_cache.count() > 0:
        vetor_busca = modelo_embedding.encode([pergunta], normalize_embeddings=True).tolist()
        res_cache = colecao_cache.query(
            query_embeddings=vetor_busca,
            n_results=1,
            where={"cliente_id": cliente_id}
        )
        if res_cache["ids"] and len(res_cache["ids"][0]) > 0:
            distancia = res_cache["distances"][0][0]
            similaridade = 1.0 - distancia
            if similaridade >= 0.90:
                dados_salvos = json.loads(res_cache["metadatas"][0][0]["resposta_json"])
                return {
                    "interacao_id": None,
                    "veio_do_cache": True,
                    "similaridade": round(similaridade * 100, 2),
                    "exibir_botoes_feedback": False,
                    "dados": dados_salvos
                }

    # 2. Se não está no cache, gera com Qwen3 + RAG
    resposta_gerada = customizar_qa_para_cliente(
        pergunta_base=pergunta,
        resposta_base=req.resposta_base,
        perfil_cliente=req.perfil_cliente.model_dump(),
        modelo_emb=modelo_embedding,
        colecao_site=colecao_site
    )

    interacao_id = f"int_{uuid.uuid4().hex[:10]}"
    pendentes_feedback[interacao_id] = {
        "pergunta": pergunta,
        "cliente_id": cliente_id,
        "resposta": resposta_gerada,
        "criado_em": datetime.now().isoformat()
    }

    return {
        "interacao_id": interacao_id,
        "veio_do_cache": False,
        "exibir_botoes_feedback": True,
        "dados": resposta_gerada
    }

@app.post("/feedback")
def endpoint_feedback(req: RequisicaoFeedback):
    item = pendentes_feedback.get(req.interacao_id)
    if not item:
        raise HTTPException(status_code=404, detail="Interação não encontrada ou já avaliada.")

    if req.aprovado:
        vetor = modelo_embedding.encode([item["pergunta"]], normalize_embeddings=True).tolist()
        id_cache = f"cache_{item['cliente_id']}_{req.interacao_id}"

        colecao_cache.upsert(
            ids=[id_cache],
            documents=[item["pergunta"]],
            embeddings=vetor,
            metadatas=[{
                "cliente_id": item["cliente_id"],
                "resposta_json": json.dumps(item["resposta"], ensure_ascii=False),
                "data_aprovacao": datetime.now().isoformat()
            }]
        )
        del pendentes_feedback[req.interacao_id]
        return {"status": "sucesso", "acao": "Resposta aprovada pelo cliente e salva no Cache Semântico!"}
    else:
        historico = []
        if os.path.exists(CAMINHO_LOG_REPROVADAS):
            with open(CAMINHO_LOG_REPROVADAS, "r", encoding="utf-8") as f:
                historico = json.load(f)

        historico.append({
            **item,
            "comentario_cliente": req.comentario_cliente,
            "data_reprovacao": datetime.now().isoformat()
        })
        with open(CAMINHO_LOG_REPROVADAS, "w", encoding="utf-8") as f:
            json.dump(historico, f, ensure_ascii=False, indent=2)

        del pendentes_feedback[req.interacao_id]
        return {"status": "reprovado", "acao": "Resposta bloqueada do cache e enviada para revisão."}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [8]:
from fastapi.testclient import TestClient

# Usa o 'app' definido na célula anterior
client = TestClient(app)

print("Testando o endpoint /status_gpu...")
response = client.get("/status_gpu")

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    print("\nResposta JSON retornada pelo endpoint:")
    import json
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))
else:
    print(f"Erro na requisição: {response.text}")

Testando o endpoint /status_gpu...
Status Code: 200

Resposta JSON retornada pelo endpoint:
{
  "gpu": "NVIDIA L4 (L4)",
  "unidades_por_hora": 4.82,
  "horas_estimadas_200_creditos": 41.5,
  "respostas_aprovadas_no_cache": 0
}
